In [2]:
import os
from typing import List, Dict, Any
import pandas as pd

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Setup completed")

/Users/macbookpro/Documents/Projects/Rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup completed


### Understanding Document Structure in LangChain

In [4]:
## create a simple document
doc  = Document(
    page_content = "This is the actual content of the document",
    metadata= {
        "source": "Source file ",
        "page":1,
        "author":"Hassan Nawaz",
        "date_created" : "2026-01-01",
        "custom_field":"here yours custom fields"
    }
)
print("Document Structure")
print(f"Content:{doc.page_content}")
print(f"Metadata:{doc.metadata}")

## Why metadata matters
print("Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in response")
print("- Debugging and auditing")

Document Structure
Content:This is the actual content of the document
Metadata:{'source': 'Source file ', 'page': 1, 'author': 'Hassan Nawaz', 'date_created': '2026-01-01', 'custom_field': 'here yours custom fields'}
Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in response
- Debugging and auditing


In [4]:
type(doc)

langchain_core.documents.base.Document

## Text Files (.txt). Simplest case 

In [5]:
import os 
os.makedirs("data/text_files", exist_ok=True)

In [16]:
sample_texts = {
    "data/text_files/python_intro.txt": """Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its simple, readable syntax and a large standard library.
Python is widely used in web development, data science, and automation.
Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its simple, readable syntax and a large standard library.
Python is widely used in web development, data science, and automation.

The language focuses on improving developer productivity and making code easier to understand.
Its clean syntax allows beginners to learn programming concepts quickly while also providing powerful features for experienced developers.
Python supports multiple programming paradigms, including object-oriented programming, procedural programming, and functional programming.

Python has a large ecosystem of libraries and frameworks that help developers build different types of applications.
Frameworks such as Django and Flask are commonly used for web development, while libraries like NumPy, Pandas, and Matplotlib are widely used for data analysis and visualization.

In artificial intelligence and machine learning, Python has become one of the most popular programming languages because of its extensive collection of tools and frameworks.
Libraries such as TensorFlow, PyTorch, and Scikit-learn allow developers to create and train machine learning models efficiently.

Python is also commonly used for automation tasks, scripting, and software testing.
Developers use Python scripts to automate repetitive tasks, manage files, process data, and interact with different systems.

Another important feature of Python is its strong community support.
Millions of developers contribute to open-source projects, create tutorials, and develop new libraries that expand Python's capabilities.

Because of its simplicity, flexibility, and powerful ecosystem, Python continues to be one of the most widely adopted programming languages in modern software development.
""",

    "data/text_files/machine_learning.txt": """Machine Learning (ML) is a branch of Artificial Intelligence (AI) that allows computers to learn from data and make predictions or decisions without being explicitly programmed for every situation.

For example, if we want a computer to identify whether an email is spam or not spam, instead of writing thousands of rules, we can provide the model with many examples of spam and normal emails. The machine learning algorithm finds patterns in the data and uses those patterns to classify new emails.

Machine Learning is commonly divided into three main types:

Supervised Learning — The model learns from labeled data.
Example: Predicting house prices.
Example: Email spam detection.
Unsupervised Learning — The model finds patterns or groups in data without predefined labels.
Example: Customer segmentation.
Example: Grouping similar products.
Reinforcement Learning — The model learns by interacting with an environment and receiving rewards or penalties.
Example: Training a game-playing AI.
Example: Robot navigation.

A typical Machine Learning workflow looks like:

Data → Preprocessing → Training → Evaluation → Prediction

The important idea is that the quality of the data strongly affects the quality of the model. A powerful algorithm cannot compensate for poor or incorrect data.
Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its simple, readable syntax and a large standard library.
Python is widely used in web development, data science, and automation.

The language focuses on improving developer productivity and making code easier to understand.
Its clean syntax allows beginners to learn programming concepts quickly while also providing powerful features for experienced developers.
Python supports multiple programming paradigms, including object-oriented programming, procedural programming, and functional programming.

Python has a large ecosystem of libraries and frameworks that help developers build different types of applications.
Frameworks such as Django and Flask are commonly used for web development, while libraries like NumPy, Pandas, and Matplotlib are widely used for data analysis and visualization.

In artificial intelligence and machine learning, Python has become one of the most popular programming languages because of its extensive collection of tools and frameworks.
Libraries such as TensorFlow, PyTorch, and Scikit-learn allow developers to create and train machine learning models efficiently.

Python is also commonly used for automation tasks, scripting, and software testing.
Developers use Python scripts to automate repetitive tasks, manage files, process data, and interact with different systems.

Another important feature of Python is its strong community support.
Millions of developers contribute to open-source projects, create tutorials, and develop new libraries that expand Python's capabilities.

Because of its simplicity, flexibility, and powerful ecosystem, Python continues to be one of the most widely adopted programming languages in modern software development."""

}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created:", list(sample_texts)) 

Sample text files created: ['data/text_files/python_intro.txt', 'data/text_files/machine_learning.txt']


### Text Loader (Able to read single file)

In [17]:
from langchain_community.document_loaders import TextLoader

## Loading a single text file
loader  = TextLoader("data/text_files/python_intro.txt" , encoding = "utf-8")

documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(f"Content preview: {documents[0].page_content[:150]}...")
print(f"Meta Data: {documents[0].metadata} ")



Loaded 1 documents
Content preview: Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its simple, readable syntax and a large standard libr...
Meta Data: {'source': 'data/text_files/python_intro.txt'} 


### DirectoryLoader- (Read multiple files)

In [18]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

## load all teext files from a directory

dir_loader = DirectoryLoader(
    "data/text_files", ## pattren to match files
    glob="**/*.txt",  ## loader class to use for loading files
    loader_cls= TextLoader,
    loader_kwargs={'encoding': "utf-8"},
    show_progress = True
)

document = dir_loader.load()
print (f"Loaded {len (document)} documents")
for i, doc in enumerate (document):
 print (f"\nDocument {i+1}: ")
 print (f"Source: {doc.metadata['source']}")
 print (f"Length: {len(doc.page_content)} characters")

 ## Analysis
print("\nDirectoryLoader Characteristics:")

print("Advantages:")
print(" - Loads multiple files at once")
print(" - Supports glob patterns")
print(" - Progress tracking")
print(" - Recursive directory scanning")

print("\nDisadvantages:")
print(" - All files must be same type")
print(" - Limited error handling per file")
print(" - Can be memory intensive for large directories")


100%|██████████| 2/2 [00:00<00:00, 968.44it/s]

Loaded 2 documents

Document 1: 
Source: data/text_files/python_intro.txt
Length: 2021 characters

Document 2: 
Source: data/text_files/machine_learning.txt
Length: 3087 characters

DirectoryLoader Characteristics:
Advantages:
 - Loads multiple files at once
 - Supports glob patterns
 - Progress tracking
 - Recursive directory scanning

Disadvantages:
 - All files must be same type
 - Limited error handling per file
 - Can be memory intensive for large directories


### Text Splitting Strategy

In [19]:
### Text splitting strategies 
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
print(document[0].page_content)


Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its simple, readable syntax and a large standard library.
Python is widely used in web development, data science, and automation.
Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its simple, readable syntax and a large standard library.
Python is widely used in web development, data science, and automation.

The language focuses on improving developer productivity and making code easier to understand.
Its clean syntax allows beginners to learn programming concepts quickly while also providing powerful features for experienced developers.
Python supports multiple programming paradigms, including object-oriented programming, procedural programming, and functional programming.

Python has a large ecosystem of libraries and frameworks that help developers build different types of applications.
Frameworks such as Django and Flask are commonly used 

In [22]:
### Method 1: Character Text Splitter
print("Character text splitter")
char_text_splitter = CharacterTextSplitter(
    separator = "\n",  # split on new line
    chunk_size = 100,   # maximum chunk size in character
    chunk_overlap = 20, # overlap between chunks
    length_function = len # how to measure chunk size
)
char_chunks = char_text_splitter.split_text(document[0].page_content)
print(f"created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

Created a chunk of size 138, which is longer than the specified 100
Created a chunk of size 138, which is longer than the specified 100
Created a chunk of size 116, which is longer than the specified 100
Created a chunk of size 178, which is longer than the specified 100
Created a chunk of size 173, which is longer than the specified 100
Created a chunk of size 129, which is longer than the specified 100
Created a chunk of size 124, which is longer than the specified 100
Created a chunk of size 137, which is longer than the specified 100


Character text splitter
created 18 chunks
First chunk: Python is a high-level programming language created by Guido van Rossum in 1991....


In [23]:
print(char_chunks[0])
print("-----------------")
print(char_chunks[1])
print("-----------------")
print(char_chunks[2])
print("-----------------")
print(char_chunks[3])
print("-----------------")
print(char_chunks[4])
print("-----------------")
print(char_chunks[5])
print("-----------------")
print(char_chunks[7])

Python is a high-level programming language created by Guido van Rossum in 1991.
-----------------
It is known for its simple, readable syntax and a large standard library.
-----------------
Python is widely used in web development, data science, and automation.
-----------------
Python is a high-level programming language created by Guido van Rossum in 1991.
-----------------
It is known for its simple, readable syntax and a large standard library.
-----------------
Python is widely used in web development, data science, and automation.
-----------------
Its clean syntax allows beginners to learn programming concepts quickly while also providing powerful features for experienced developers.


In [28]:
### Recursive Character Text Splitter
print("Recursive Character Text Splitter")
recursive_text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " " , ""], ## try these separators in order.
    chunk_size=200,
    chunk_overlap=20,
    length_function = len
)
recursive_chunks     = recursive_text_splitter.split_text(document[0].page_content)
print(f"created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")

Recursive Character Text Splitter
created 12 chunks
First chunk: Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its...


In [29]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])
print("------------------")
print(recursive_chunks[2])
print("------------------")
print(recursive_chunks[3])
print("------------------")
print(recursive_chunks[4])
print("------------------")
print(recursive_chunks[5])


Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its simple, readable syntax and a large standard library.
Python is widely used in web development,
-----------------
in web development, data science, and automation.
Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its simple, readable syntax and a large standard
------------------
a large standard library.
Python is widely used in web development, data science, and automation.

The language focuses on improving developer productivity and making code easier to understand.
Its
------------------
to understand.
Its clean syntax allows beginners to learn programming concepts quickly while also providing powerful features for experienced developers.
Python supports multiple programming
------------------
programming paradigms, including object-oriented programming, procedural programming, and functional programming.

Python has a large ecosystem

In [33]:
## Method 3: Token Text Splitter
print("Token text splitter")
token_text_splitter = TokenTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    length_function = len
)
token_chunks = token_text_splitter.split_text(document[0].page_content)
print(f"created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")

Token text splitter
created 2 chunks
First chunk: Python is a high-level programming language created by Guido van Rossum in 1991.
It is known for its...


In [34]:
# 📊 Comparison

print("\n📊 Text Splitting Methods Comparison:")

print("\nCharacterTextSplitter:")
print("  ✅ Simple and predictable")
print("  ✅ Good for structured text")
print("  ❌ May break mid-sentence")
print("  ➡️ Use when: Text has clear delimiters")


print("\nRecursiveCharacterTextSplitter:")
print("  ✅ Respects text structure")
print("  ✅ Tries multiple separators")
print("  ✅ Best general-purpose splitter")
print("  ❌ Slightly more complex")
print("  ➡️ Use when: Default choice for most texts")


print("\nTokenTextSplitter:")
print("  ✅ Respects model token limits")
print("  ✅ More accurate for embeddings")
print("  ❌ Slower than character-based")
print("  ➡️ Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
  ✅ Simple and predictable
  ✅ Good for structured text
  ❌ May break mid-sentence
  ➡️ Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
  ✅ Respects text structure
  ✅ Tries multiple separators
  ✅ Best general-purpose splitter
  ❌ Slightly more complex
  ➡️ Use when: Default choice for most texts

TokenTextSplitter:
  ✅ Respects model token limits
  ✅ More accurate for embeddings
  ❌ Slower than character-based
  ➡️ Use when: Working with token-limited models
